In [1]:
# Merge per-video ASR .npy → embeddings.npy + gallery_map.csv (same schema as CLIP).
# n_rows = number of segments; row i in the slice ↔ line i of VIDEO_ID.jsonl.
from pathlib import Path

ASR_DIR = Path(r"d:/HCMUS_ComputerScience/code/AIC2026/features/asr_emb")
# ASR_DIR = Path(r"C:/AIC2026-media/features/asr_emb")

OUT_NPY = ASR_DIR / "embeddings.npy"
OUT_MAP = ASR_DIR / "gallery_map.csv"  # video_id, start_row, n_rows
OVERWRITE = False

print("ASR_DIR =", ASR_DIR.resolve())
print("OUT_NPY =", OUT_NPY)
print("OUT_MAP =", OUT_MAP)


ASR_DIR = D:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb
OUT_NPY = d:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb\embeddings.npy
OUT_MAP = d:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb\gallery_map.csv


In [2]:
import csv

import numpy as np

if not ASR_DIR.is_dir():
    raise SystemExit(f"Not a directory: {ASR_DIR}")

npy_paths = sorted(ASR_DIR.glob("*/*.npy"))
if not npy_paths:
    npy_paths = sorted(ASR_DIR.glob("*.npy"))
npy_paths = [p for p in npy_paths if p.name != "embeddings.npy"]
if not npy_paths:
    raise SystemExit(f"No per-video .npy under {ASR_DIR}")

print(f"videos={len(npy_paths)}")

if OUT_NPY.is_file() and OUT_MAP.is_file() and not OVERWRITE:
    emb = np.load(OUT_NPY)
    print(f"Skip merge (exists). embeddings.npy shape={emb.shape} dtype={emb.dtype}")
    print(f"gallery_map.csv exists: {OUT_MAP}")
else:
    blocks = []
    rows = []
    start = 0
    for p in npy_paths:
        block = np.load(p).astype(np.float32, copy=False)
        if block.ndim != 2:
            raise SystemExit(f"Expected 2D {p}, got {block.shape}")
        n = int(block.shape[0])
        jsonl = p.with_suffix(".jsonl")
        if jsonl.is_file():
            n_lines = sum(1 for line in jsonl.read_text(encoding="utf-8").splitlines() if line.strip())
            if n_lines != n:
                print(f"warning: {p.stem} npy={n} jsonl={n_lines}", flush=True)
        blocks.append(block)
        rows.append({"video_id": p.stem, "start_row": start, "n_rows": n})
        start += n
        print(f"  {p.parent.name}/{p.name}  segs={n}", flush=True)

    emb = np.concatenate(blocks, axis=0)
    np.save(OUT_NPY, emb)
    with OUT_MAP.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["video_id", "start_row", "n_rows"])
        w.writeheader()
        w.writerows(rows)

    print(f"Wrote {OUT_NPY}  shape={emb.shape} dtype={emb.dtype}")
    print(f"Wrote {OUT_MAP}  ({len(rows)} videos, total_rows={start})")


videos=114
Skip merge (exists). embeddings.npy shape=(23171, 384) dtype=float32
gallery_map.csv exists: d:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb\gallery_map.csv


In [3]:
import csv

emb = np.load(OUT_NPY)
with OUT_MAP.open(encoding="utf-8", newline="") as f:
    map_rows = list(csv.DictReader(f))

covered = sum(int(r["n_rows"]) for r in map_rows)
print("shape =", emb.shape)
print("map videos =", len(map_rows), " covered rows =", covered)
assert covered == emb.shape[0], "gallery_map row count != embeddings rows"
assert emb.shape[-1] == 384, f"expected dim 384, got {emb.shape[-1]}"
print("OK")


shape = (23171, 384)
map videos = 114  covered rows = 23171
OK
